In [4]:
import ollama
from ollama import Client

In [5]:
client = Client(host="http://localhost:11434")

In [ ]:
IMAGE_PATH = "./data/img3.png"   # your extracted frame

with open(IMAGE_PATH, "rb") as f:
    image_bytes = f.read()

prompt = """
You are an OCR and data-extraction expert.

Focus ONLY on the fencing scoreboard overlay located at the BOTTOM 20% of the image.
Ignore everything else in the image, including flags, country codes, uniforms, and background action.

The overlay typically contains (from left to right):
- A country code and flag (IGNORE these)
- The left fencer's name
- The left fencer's score
- A timer (IGNORE)
- The right fencer's score
- The right fencer's name
- A country code and flag (IGNORE these)

Names can be full names or abbreviations, for example:
- "Svichkar R."
- "Di Veroli D."
- "KANO K."
- "Gergely S."
- "Lee"
Do NOT assume a fixed pattern like LASTNAME + INITIAL only.

Important rules for country codes vs names:
- Country codes are usually 2-3 letters in ALL CAPS (e.g., "UKR", "ITA", "USA", "FRA", "HUN", "GER", "CHN", "KOR").
- If a text is exactly 2-3 uppercase letters, treat it as a country code, NOT a name, and ignore it.
- Prefer the longest, most name-like string near each score (may include spaces, lowercase letters, or periods).
- If you accidentally pick a country code as a name, you must correct yourself before responding.

Your output must be ONLY this JSON object:

{
  "left_name": string or null,
  "left_score": integer or null,
  "right_name": string or null,
  "right_score": integer or null
}

Additional rules:
- Use the exact spelling of the names as shown on the scoreboard overlay.
- Scores must be integers.
- If you cannot confidently read a value, return null for that field.
- Output ONLY the JSON object with no extra text before or after.
"""

model = "llama3.2-vision"
# model = "llava:7b"

resp = client.chat(
    model=model,
    messages=[
        {
            "role": "user",
            "content": prompt,
            "images": [image_bytes]
        }
    ]
)

print(resp["message"]["content"])


{
  "left_name": "Gergely S.",
  "left_score": 5,
  "right_name": "KANO K.",
  "right_score": 6
}


In [16]:
images = ["img0.png", "img1.png", "img2.png", "img3.png"]
for image in images:
    IMAGE_PATH = f"./data/{image}"   # your extracted frame

    with open(IMAGE_PATH, "rb") as f:
        image_bytes = f.read()

    resp = client.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt,
                "images": [image_bytes]
            }
        ]
    )

    print(f"Results for {image}:")
    print(resp["message"]["content"])
    print()

Results for img0.png:
{
  "left_name": "Svichkar R.",
  "left_score": 3,
  "right_name": "Di Veroli D.",
  "right_score": 3
}

Results for img1.png:
{
  "left_name": "Takahiro SHIKINE",
  "left_score": 0,
  "right_name": "Gerek MEINHARDT",
  "right_score": 0
}

Results for img2.png:
{
  "left_name": "Ruben LIMARDO GASCON",
  "left_score": null,
  "right_name": null,
  "right_score": null
}

Results for img3.png:
{
  "left_name": "Gergely S.",
  "left_score": 5,
  "right_name": "KANO K.",
  "right_score": 6
}

